## Objective
This notebook brings together the performance, risk, scenario, and stress-testing results developed throughout the project to evaluate the three strategic portfolio allocations.

The objective is to move from measuring risk to making portfolio-management decisions. Each strategy is assessed against its intended purpose, with particular attention to whether its historical performance, downside exposure, and sensitivity to market risks remain consistent with that purpose.

Where a meaningful risk concentration is identified, a potential allocation adjustment is evaluated. The notebook then develops a monitoring and rebalancing framework and concludes with a final assessment of the three strategies.

## Analysis Framework

The portfolio evaluation is organized into five stages:

1. **Portfolio Risk Assessment** — Synthesize the major performance and risk findings from the previous notebooks.
2. **Risk Objectives and Constraints** — Define the intended role and acceptable risk characteristics of each portfolio.
3. **Allocation and Rebalancing Evaluation** — Determine whether the existing strategic allocations remain consistent with their objectives.
4. **Portfolio Monitoring Framework** — Establish indicators and conditions that would trigger review or rebalancing.
5. **Final Portfolio Assessment** — Compare the three strategies and summarize the major portfolio-management conclusions.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths to previously saved project outputs
n5_output_dir = Path("notebook_5_outputs")

# Load Notebook 5 reverse stress results
reverse_stress_equity = pd.read_csv(
    n5_output_dir / "reverse_stress_equity.csv"
)

reverse_stress_comparison = pd.read_csv(
    n5_output_dir / "reverse_stress_comparison.csv"
)

reverse_stress_scenarios = pd.read_csv(
    n5_output_dir / "reverse_stress_scenarios.csv"
)

print("Notebook 5 stress-testing results loaded successfully.")

Notebook 5 stress-testing results loaded successfully.


In [2]:
# Load historical portfolio returns from Notebook 2
portfolio_returns = pd.read_csv(
    "portfolio_returns.csv",
    index_col=0,
    parse_dates=True
)

# Load historical asset returns from Notebook 1
asset_returns = pd.read_csv(
    "returns_daily.csv",
    index_col=0,
    parse_dates=True
)

# Display historical sample information
print(
    f"Historical sample: "
    f"{portfolio_returns.index.min().date()} to "
    f"{portfolio_returns.index.max().date()}"
)

print(
    f"Trading-day observations: "
    f"{len(portfolio_returns):,}"
)

Historical sample: 2007-05-31 to 2026-09-18
Trading-day observations: 4,857


## Part I — Portfolio Risk Assessment
The evaluation begins by consolidating the main findings from the previous notebooks. Historical return and risk measures are considered alongside stress-test results to identify the primary strengths and vulnerabilities of each strategy.

### Core Portfolio Risk and Performance Summary

Annualized return and volatility provide a long-term view of performance and variability, while maximum drawdown, Value at Risk (VaR), and Expected Shortfall (ES) focus more directly on downside risk.

These measures are recalculated from the portfolio return series so the final assessment uses a consistent historical sample.

In [3]:
# Core portfolio risk and performance metrics

trading_days = 252
confidence_level = 0.95

risk_summary = []

for portfolio in portfolio_returns.columns:

    returns = portfolio_returns[portfolio].dropna()

    # Annualized compound return
    annualized_return = (
        (1 + returns).prod() ** (trading_days / len(returns))
        - 1
    )

    # Annualized volatility
    annualized_volatility = (
        returns.std() * np.sqrt(trading_days)
    )

    # Maximum drawdown
    cumulative_growth = (1 + returns).cumprod()
    running_peak = cumulative_growth.cummax()

    drawdown = (
        cumulative_growth / running_peak - 1
    )

    max_drawdown = drawdown.min()

    # Historical 95% VaR
    var_cutoff = returns.quantile(
        1 - confidence_level
    )

    historical_var = -var_cutoff

    # Historical 95% Expected Shortfall
    historical_es = -returns[
        returns <= var_cutoff
    ].mean()

    risk_summary.append({
        "Portfolio": portfolio,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_volatility,
        "Maximum Drawdown": max_drawdown,
        "Daily VaR (95%)": historical_var,
        "Daily ES (95%)": historical_es
    })

risk_summary = pd.DataFrame(risk_summary)
# Calculate Sharpe using the BIL risk-free proxy
risk_free_returns = asset_returns["BIL"].reindex(
    portfolio_returns.index
)

if risk_free_returns.isna().any():
    raise ValueError(
        "BIL returns are missing from the portfolio sample."
    )

# Calculate daily excess portfolio returns
excess_returns = portfolio_returns.sub(
    risk_free_returns,
    axis=0
)

# Calculate annualized Sharpe Ratios
sharpe_ratios = (
    excess_returns.mean()
    / excess_returns.std()
    * np.sqrt(trading_days)
)

# Add Sharpe Ratios to the existing summary table
risk_summary["Sharpe Ratio"] = (
    risk_summary["Portfolio"]
    .map(sharpe_ratios)
)

display(
    risk_summary.style
    .format({
        "Annualized Return": "{:.2%}",
        "Annualized Volatility": "{:.2%}",
        "Maximum Drawdown": "{:.2%}",
        "Daily VaR (95%)": "{:.2%}",
        "Daily ES (95%)": "{:.2%}",
        "Sharpe Ratio": "{:.2f}"
    })
    .hide(axis="index")
    .set_caption(
        "Core Portfolio Risk and Performance Summary"
    )
)

Portfolio,Annualized Return,Annualized Volatility,Maximum Drawdown,Daily VaR (95%),Daily ES (95%),Sharpe Ratio
Aggressive,8.47%,17.43%,-50.09%,1.59%,2.66%,0.47
Balanced,7.43%,13.56%,-39.49%,1.23%,2.04%,0.49
Resilient,6.48%,9.00%,-25.77%,0.81%,1.32%,0.59


#### Interpretation
The historical results show a clear trade-off across the three allocations. Aggressive Growth generated the highest annualized return, but it also experienced the greatest volatility, maximum drawdown, and daily tail-loss exposure. Balanced Growth remained between the two strategies, while Resilient Growth provided the strongest historical downside protection.

The Sharpe Ratio adds another perspective by considering excess return relative to volatility. The original Resilient portfolio achieved the highest historical Sharpe, showing that its lower return was accompanied by a proportionally larger reduction in return variability.

These measures establish the portfolios' historical risk-return characteristics, but they do not fully describe how each strategy responds to different market shocks. The stress-testing results provide additional evidence for the allocation decisions that follow.

### Stress and Scenario Risk Assessment
The historical risk measures are complemented by the scenario, sensitivity, and reverse stress-testing results from Notebooks 4 and 5.

The comparison focuses on each portfolio's most adverse hypothetical scenario, its model-based response to the 2008 and 2022 historical stress directions, and the market shocks required to reach a 20% portfolio-loss threshold.

Together, these results help identify whether the more defensive allocations continue to provide protection across different sources of market stress.


#### Consolidated Stress-Risk Summary


In [4]:
# Portfolio names used throughout Notebook 6
portfolio_columns = portfolio_returns.columns.tolist()

portfolio_columns

# Load hypothetical stress returns calculated in Notebook 5
hypothetical_stress = pd.read_csv(
    n5_output_dir / "hypothetical_stress_results.csv",
    index_col="Portfolio"
).reindex(portfolio_columns)

if hypothetical_stress.isna().any().any():
    raise ValueError(
        "Hypothetical stress results are missing for one or more portfolios."
    )

# Identify the most adverse hypothetical scenario
worst_scenario = hypothetical_stress.idxmin(axis=1)

worst_scenario_loss = (
    hypothetical_stress.min(axis=1).abs()
)


# Equity decline required to produce a 20% portfolio loss
equity_20 = (
    reverse_stress_equity[
        np.isclose(
            reverse_stress_equity["Loss Threshold"],
            0.20
        )
    ]
    .set_index("Portfolio")["Required Equity Shock"]
)


# Allow for either version of the column name
if "20% Loss Scale" in reverse_stress_comparison.columns:
    scale_column = "20% Loss Scale"
else:
    scale_column = "20% Loss"


# Extract 2008 reverse-stress results
stress_2008 = (
    reverse_stress_comparison[
        reverse_stress_comparison[
            "Historical Direction"
        ].str.contains("2008", case=False)
    ]
    .set_index("Portfolio")
)


# Extract 2022 reverse-stress results
stress_2022 = (
    reverse_stress_comparison[
        reverse_stress_comparison[
            "Historical Direction"
        ].str.contains("2022", case=False)
    ]
    .set_index("Portfolio")
)

# Construct consolidated stress-risk summary
stress_risk_summary = pd.DataFrame(
    index=portfolio_columns
)

stress_risk_summary["Worst Hypothetical Stress"] = (
    worst_scenario
)

stress_risk_summary["Worst Hypothetical Loss"] = (
    worst_scenario_loss
)

stress_risk_summary["2008 Model-Based Loss"] = (
    stress_2008["Original Model-Based Loss"]
)

stress_risk_summary["2022 Model-Based Loss"] = (
    stress_2022["Original Model-Based Loss"]
)

stress_risk_summary[
    "Equity Shock for 20% Loss"
] = equity_20

stress_risk_summary[
    "2008 Scale for 20% Loss"
] = stress_2008[scale_column]

stress_risk_summary[
    "2022 Scale for 20% Loss"
] = stress_2022[scale_column]


# Move portfolio names into their own column
stress_risk_summary = (
    stress_risk_summary
    .rename_axis("Portfolio")
    .reset_index()
)


display(
    stress_risk_summary.style
    .format({
        "Worst Hypothetical Loss": "{:.2%}",
        "2008 Model-Based Loss": "{:.2%}",
        "2022 Model-Based Loss": "{:.2%}",
        "Equity Shock for 20% Loss": "{:.2%}",
        "2008 Scale for 20% Loss": "{:.2f}x",
        "2022 Scale for 20% Loss": "{:.2f}x"
    })
    .hide(axis="index")
    .set_caption(
        "Consolidated Portfolio Stress-Risk Assessment"
    )
)

Portfolio,Worst Hypothetical Stress,Worst Hypothetical Loss,2008 Model-Based Loss,2022 Model-Based Loss,Equity Shock for 20% Loss,2008 Scale for 20% Loss,2022 Scale for 20% Loss
Aggressive,Severe Global Recession,37.15%,26.79%,17.75%,-24.45%,0.75x,1.13x
Balanced,Severe Global Recession,27.80%,21.53%,16.30%,-31.57%,0.93x,1.23x
Resilient,Severe Stagflation,15.90%,14.92%,14.47%,-49.92%,1.34x,1.38x


#### Interpretation
The consolidated stress assessment shows that Aggressive Growth remains most exposed to severe equity-led losses, while Resilient Growth generally requires larger adverse equity movements to reach the same portfolio-loss threshold.

The most adverse hypothetical scenario also changes with allocation. Severe Global Recession produces the largest loss for Aggressive and Balanced Growth, while Severe Stagflation produces the largest loss for Resilient Growth. This reflects the increasing importance of bond and interest-rate exposure as the portfolio becomes more defensive.  A severe global recession produces the largest hypothetical loss for the Aggressive and Balanced portfolios, reflecting their greater equity exposure. In contrast, severe stagflation produces the largest hypothetical loss for Resilient Growth, highlighting the increasing importance of interest-rate and inflation-related risks as fixed-income allocations rise.

The historical multi-factor stress results reinforce this distinction. The 2008 financial-crisis direction produces particularly severe losses for the equity-oriented portfolios, while the 2022 inflation and rate-shock direction narrows the differences between the three strategies. Reducing equity exposure improves resilience to some shocks, but it changes the portfolio's other vulnerabilities rather than eliminating risk.

## Part II — Risk Objectives and Portfolio Constraints
Each portfolio is evaluated according to its intended strategic role rather than a single definition of acceptable risk.

The objectives below are analytical mandates developed for this project. They provide a basis for determining whether the portfolios' historical performance, downside exposure, and stress sensitivities remain consistent with their intended purpose.

### Portfolio Objectives

In [5]:
portfolio_objectives = pd.DataFrame({
    "Portfolio": [
        "Aggressive",
        "Balanced",
        "Resilient"
    ],
    "Primary Objective": [
        "Maximize long-term capital growth",
        "Balance long-term growth with downside protection",
        "Emphasize downside protection while maintaining long-term growth exposure"
    ],
    "Intended Risk Posture": [
        "High",
        "Moderate",
        "Defensive"
    ],
    "Primary Risk Trade-Off": [
        "Higher return potential in exchange for greater equity-driven volatility and drawdown risk",
        "Reduced downside risk while retaining meaningful participation in equity growth",
        "Greater downside protection in exchange for lower return potential and increased interest-rate sensitivity"
    ]
})

display(
    portfolio_objectives.style
    .hide(axis="index")
    .set_caption(
        "Strategic Portfolio Objectives"
    )
)

Portfolio,Primary Objective,Intended Risk Posture,Primary Risk Trade-Off
Aggressive,Maximize long-term capital growth,High,Higher return potential in exchange for greater equity-driven volatility and drawdown risk
Balanced,Balance long-term growth with downside protection,Moderate,Reduced downside risk while retaining meaningful participation in equity growth
Resilient,Emphasize downside protection while maintaining long-term growth exposure,Defensive,Greater downside protection in exchange for lower return potential and increased interest-rate sensitivity


### Risk Constraint Framework
The risk framework distinguishes between portfolio characteristics that should remain consistent with the strategic design and risk indicators that warrant further review.

Structural constraints address the intended allocation and portfolio construction. Monitoring indicators track whether volatility, downside losses, stress exposure, or factor sensitivity have changed in a way that may no longer be consistent with the portfolio's objective.

A deterioration in a risk measure triggers investigation rather than automatically requiring a trade. The measures are evaluated relative to each portfolio's stated objective and to the other strategies rather than against a single universal risk threshold.


In [6]:
risk_constraint_framework = pd.DataFrame({
    "Risk Area": [
        "Strategic Allocation",
        "Portfolio Volatility",
        "Downside Risk",
        "Stress-Test Loss",
        "Equity Exposure",
        "Interest-Rate Exposure",
        "Allocation Drift"
    ],

    "Constraint Type": [
        "Structural",
        "Monitoring",
        "Monitoring",
        "Monitoring",
        "Monitoring",
        "Monitoring",
        "Rebalancing"
    ],

    "Assessment": [
        "Maintain the intended strategic asset-class structure",
        "Confirm that realized volatility remains consistent with the portfolio's intended risk posture",
        "Monitor drawdown, VaR, and ES for deterioration relative to historical behavior",
        "Evaluate whether adverse scenarios produce losses inconsistent with the portfolio's intended resilience",
        "Monitor equity sensitivity relative to the portfolio's growth objective",
        "Monitor increasing rate sensitivity as fixed-income exposure rises",
        "Review the portfolio when asset weights move materially away from strategic targets"
    ],

    "Management Response": [
        "Review material changes to strategic weights",
        "Investigate sustained changes in portfolio volatility",
        "Assess the source of unusually large downside losses",
        "Review allocation if stress vulnerability materially increases",
        "Evaluate whether equity risk remains appropriate for the strategy",
        "Evaluate whether defensive allocation creates excessive rate exposure",
        "Rebalance toward strategic target weights when predefined bands are breached"
    ]
})

display(
    risk_constraint_framework.style
    .hide(axis="index")
    .set_caption("Portfolio Risk Constraint Framework")
)

Risk Area,Constraint Type,Assessment,Management Response
Strategic Allocation,Structural,Maintain the intended strategic asset-class structure,Review material changes to strategic weights
Portfolio Volatility,Monitoring,Confirm that realized volatility remains consistent with the portfolio's intended risk posture,Investigate sustained changes in portfolio volatility
Downside Risk,Monitoring,"Monitor drawdown, VaR, and ES for deterioration relative to historical behavior",Assess the source of unusually large downside losses
Stress-Test Loss,Monitoring,Evaluate whether adverse scenarios produce losses inconsistent with the portfolio's intended resilience,Review allocation if stress vulnerability materially increases
Equity Exposure,Monitoring,Monitor equity sensitivity relative to the portfolio's growth objective,Evaluate whether equity risk remains appropriate for the strategy
Interest-Rate Exposure,Monitoring,Monitor increasing rate sensitivity as fixed-income exposure rises,Evaluate whether defensive allocation creates excessive rate exposure
Allocation Drift,Rebalancing,Review the portfolio when asset weights move materially away from strategic targets,Rebalance toward strategic target weights when predefined bands are breached


### Portfolio Mandate Consistency Assessment
The historical risk and stress-testing evidence is used to assess whether each allocation remains consistent with its stated objective. The assessment considers expected trade-offs within each strategy rather than treating higher volatility or drawdown as an automatic reason to change the portfolio.

In [7]:
mandate_assessment = pd.DataFrame({
    "Portfolio": [
        "Aggressive",
        "Balanced",
        "Resilient"
    ],

    "Growth Profile": [
        "Highest historical return and strongest equity exposure",
        "Intermediate return and equity exposure",
        "Lowest historical return and lowest equity exposure"
    ],

    "Downside Profile": [
        "Highest volatility, drawdown, VaR, and ES",
        "Moderate downside risk across measures",
        "Lowest volatility, drawdown, VaR, and ES"
    ],

    "Stress Profile": [
        "Most vulnerable to severe equity-led stress",
        "Intermediate vulnerability across stress scenarios",
        "Strongest downside protection, but more exposed to rate-driven stress"
    ],

    "Primary Risk Trade-Off": [
        "Accepts substantial equity risk in pursuit of growth",
        "Balances equity growth with defensive allocation",
        "Reduces equity risk while accepting greater interest-rate sensitivity"
    ],

    "Mandate Assessment": [
        "Consistent",
        "Consistent",
        "Consistent — monitor rate exposure"
    ]
})

display(
    mandate_assessment.style
    .hide(axis="index")
    .set_caption(
        "Portfolio Mandate Consistency Assessment"
    )
)

Portfolio,Growth Profile,Downside Profile,Stress Profile,Primary Risk Trade-Off,Mandate Assessment
Aggressive,Highest historical return and strongest equity exposure,"Highest volatility, drawdown, VaR, and ES",Most vulnerable to severe equity-led stress,Accepts substantial equity risk in pursuit of growth,Consistent
Balanced,Intermediate return and equity exposure,Moderate downside risk across measures,Intermediate vulnerability across stress scenarios,Balances equity growth with defensive allocation,Consistent
Resilient,Lowest historical return and lowest equity exposure,"Lowest volatility, drawdown, VaR, and ES","Strongest downside protection, but more exposed to rate-driven stress",Reduces equity risk while accepting greater interest-rate sensitivity,Consistent — monitor rate exposure


#### Interpretation
All three strategies remain broadly consistent with their intended objectives. Aggressive Growth accepts greater equity-driven downside risk in pursuit of long-term growth, while Balanced Growth provides an intermediate profile across the major risk measures.

Resilient Growth provides the strongest historical downside protection, but its larger bond allocation introduces greater sensitivity to rising interest rates. This is a risk concentration worth examining, not evidence that the entire strategy has failed.

The assessment therefore supports maintaining the basic structure of the three portfolios while evaluating whether a targeted adjustment could improve the Resilient strategy's exposure to rate-driven stress. Potential allocation changes should therefore be evaluated only where they improve a specific risk trade-off without materially undermining the portfolio's strategic objective.

## Part III — Allocation and Rebalancing Evaluation

The allocation and rebalancing evaluation examines whether the existing strategic portfolio weights remain appropriate given the risk objectives and portfolio behavior identified in the preceding analysis.

Because the three strategies are currently consistent with their intended mandates, allocation changes are not assumed to be necessary. Instead, potential adjustments are considered only where the existing allocation creates a meaningful risk concentration or where a change could improve a specific risk-return trade-off without undermining the portfolio's intended purpose.

The evaluation therefore begins with the current strategic asset allocations, followed by an assessment of whether the observed equity, interest-rate, downside, and stress exposures provide sufficient justification for modifying any portfolio weights.

### Current Strategic Allocation Review

The original strategic weights are reproduced below to identify the asset exposures underlying each portfolio's historical and stress-test results.

In [8]:
# Strategic asset allocations

weights_aggressive = {
    "VTI": 0.55,
    "VEU": 0.20,
    "VWO": 0.10,
    "BND": 0.05,
    "GLD": 0.05,
    "BIL": 0.05
}

weights_balanced = {
    "VTI": 0.40,
    "VEU": 0.15,
    "VWO": 0.10,
    "BND": 0.25,
    "GLD": 0.05,
    "BIL": 0.05
}

weights_resilient = {
    "VTI": 0.25,
    "VEU": 0.10,
    "VWO": 0.05,
    "BND": 0.45,
    "GLD": 0.10,
    "BIL": 0.05
}

strategic_allocations = pd.DataFrame({
    "Aggressive": weights_aggressive,
    "Balanced": weights_balanced,
    "Resilient": weights_resilient
}).T

display(
    strategic_allocations.style
    .format("{:.0%}")
    .set_caption("Current Strategic Asset Allocations")
)

,VTI,VEU,VWO,BND,GLD,BIL
Aggressive,55%,20%,10%,5%,5%,5%
Balanced,40%,15%,10%,25%,5%,5%
Resilient,25%,10%,5%,45%,10%,5%


#### Interpretation
The three allocations progressively reduce equity exposure and increase defensive assets. Aggressive Growth has the largest equity allocation, while Resilient Growth has the greatest exposure to broad investment-grade bonds.

The preceding analyses show that this progression reduces historical equity-related downside risk but increases the importance of interest-rate sensitivity. The allocation review therefore focuses on whether the defensive assets are distributed appropriately, rather than simply asking whether each portfolio should take less risk.

### Need for Allocation Adjustment
An allocation change is considered when the existing portfolio presents a meaningful risk concentration that can be addressed without undermining its intended purpose.

The review considers whether the identified concern is an expected feature of the strategy or a vulnerability that warrants further evaluation. The purpose is not to minimize risk across all portfolios, but to determine whether the current allocation remains appropriate for the role each strategy is intended to serve.


In [9]:
allocation_review = pd.DataFrame({
    "Portfolio": [
        "Aggressive",
        "Balanced",
        "Resilient"
    ],

    "Mandate Fit": [
        "Consistent",
        "Consistent",
        "Consistent"
    ],

    "Primary Concern": [
        "High equity-driven downside risk",
        "Moderate exposure across both equity and rate risk",
        "Greater interest-rate sensitivity from large bond allocation"
    ],

    "Does Concern Conflict With Mandate?": [
        "No",
        "No",
        "Not necessarily"
    ],

    "Current Decision": [
        "Maintain",
        "Maintain",
        "Monitor / Evaluate"
    ]
})

display(
    allocation_review.style
    .hide(axis="index")
    .set_caption(
        "Strategic Allocation Review"
    )
)

Portfolio,Mandate Fit,Primary Concern,Does Concern Conflict With Mandate?,Current Decision
Aggressive,Consistent,High equity-driven downside risk,No,Maintain
Balanced,Consistent,Moderate exposure across both equity and rate risk,No,Maintain
Resilient,Consistent,Greater interest-rate sensitivity from large bond allocation,Not necessarily,Monitor / Evaluate


#### Interpretation
The evidence does not identify a specific reason to modify the Aggressive or Balanced allocations. Their principal risks remain consistent with their intended objectives.  The higher equity-related downside risk of Aggressive Growth represents an expected consequence of its growth-oriented mandate, while Balanced Growth continues to provide an intermediate risk profile across the major historical and stress measures.

Resilient Growth also remains consistent with its defensive mandate, but its larger BND allocation creates a measurable concentration in bond-duration exposure. A targeted adjustment is therefore evaluated to determine whether that concentration can be reduced without materially changing the portfolio's overall role.

### Resilient Portfolio Adjustment Evaluation

The preceding analysis identifies interest-rate sensitivity as the primary risk concentration warranting further evaluation within the Resilient Growth portfolio. The existing allocation remains consistent with its defensive mandate, so the objective is not to redesign the strategy or maximize historical performance.

Instead, a modest alternative allocation is evaluated to determine whether reducing fixed-income duration exposure can improve the portfolio's risk profile while preserving its existing equity allocation and overall defensive characteristics.

The candidate allocation reduces the allocation to broad investment-grade bonds and reallocates the difference equally to gold and short-term Treasury exposure. This adjustment is intended to reduce interest-rate sensitivity while maintaining diversification across defensive assets. The proposed weights are illustrative rather than optimized portfolio weights.

The total equity allocation remains unchanged at 40%, allowing the comparison to focus on the composition of the defensive allocation rather than a reduction in equity exposure.

In [10]:
# Original Resilient allocation
resilient_original = {
    "VTI": 0.25,
    "VEU": 0.10,
    "VWO": 0.05,
    "BND": 0.45,
    "GLD": 0.10,
    "BIL": 0.05
}

# Candidate Resilient allocation
resilient_adjusted = {
    "VTI": 0.25,
    "VEU": 0.10,
    "VWO": 0.05,
    "BND": 0.35,
    "GLD": 0.15,
    "BIL": 0.10
}

allocation_comparison = pd.DataFrame({
    "Original Resilient": resilient_original,
    "Adjusted Resilient": resilient_adjusted
}).T

display(
    allocation_comparison.style
    .format("{:.0%}")
    .set_caption("Original vs. Adjusted Resilient Allocation")
)

,VTI,VEU,VWO,BND,GLD,BIL
Original Resilient,25%,10%,5%,45%,10%,5%
Adjusted Resilient,25%,10%,5%,35%,15%,10%


In [11]:
# Identify the assets required for the adjusted Resilient portfolio
required_assets = list(resilient_adjusted.keys())

# Check that all required assets are available
missing_assets = [
    asset for asset in required_assets
    if asset not in asset_returns.columns
]

if missing_assets:
    raise ValueError(
        f"Missing asset returns: {missing_assets}"
    )

print("All required asset returns are available.")

All required asset returns are available.


In [12]:
adjusted_weights = pd.Series(resilient_adjusted)

resilient_adjusted_returns = (
    asset_returns[adjusted_weights.index]
    .dropna()
    .dot(adjusted_weights)
)

resilient_adjusted_returns.name = "Adjusted Resilient"

resilient_comparison_returns = pd.concat(
    [
        portfolio_returns["Resilient"],
        resilient_adjusted_returns
    ],
    axis=1,
    join="inner"
).dropna()

resilient_comparison_returns.columns = [
    "Original Resilient",
    "Adjusted Resilient"
]

resilient_comparison_returns.head()

,Original Resilient,Adjusted Resilient
Date,,
2007-05-31,0.001999,0.002712
2007-06-01,0.003460,0.004388
2007-06-04,0.001095,0.001074
2007-06-05,-0.003120,-0.003087
2007-06-06,-0.004277,-0.004319


#### Risk and Performance Comparison

The original and adjusted Resilient portfolios are compared over the same historical sample to determine whether the proposed reduction in fixed-income exposure materially changes the portfolio's risk-return characteristics.

The purpose of the adjustment is not to produce substantially different day-to-day returns or maximize historical performance. Instead, the analysis evaluates whether interest-rate exposure can be reduced while preserving the defensive characteristics of the original strategy.

In [13]:
# Compare historical risk and performance characteristics

trading_days = 252
confidence_level = 0.95

adjustment_metrics = []

for portfolio in resilient_comparison_returns.columns:

    returns = resilient_comparison_returns[portfolio].dropna()

    # Annualized compound return
    annualized_return = (
        (1 + returns).prod() ** (trading_days / len(returns))
        - 1
    )

    # Annualized volatility
    annualized_volatility = (
        returns.std() * np.sqrt(trading_days)
    )

    # Maximum drawdown
    cumulative_growth = (1 + returns).cumprod()
    running_peak = cumulative_growth.cummax()
    drawdown = cumulative_growth / running_peak - 1
    max_drawdown = drawdown.min()

    # Historical 95% VaR
    var_cutoff = returns.quantile(
        1 - confidence_level
    )
    historical_var = -var_cutoff

    # Historical 95% Expected Shortfall
    historical_es = -returns[
        returns <= var_cutoff
    ].mean()

    adjustment_metrics.append({
        "Portfolio": portfolio,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_volatility,
        "Maximum Drawdown": max_drawdown,
        "Daily VaR (95%)": historical_var,
        "Daily ES (95%)": historical_es
    })

adjustment_comparison = pd.DataFrame(
    adjustment_metrics
)

# Use the same BIL-based Sharpe methodology as Notebook 3
adjustment_rf = asset_returns["BIL"].reindex(
    resilient_comparison_returns.index
)

if adjustment_rf.isna().any():
    raise ValueError(
        "BIL returns are missing from the adjustment sample."
    )

adjustment_excess_returns = (
    resilient_comparison_returns
    .sub(adjustment_rf, axis=0)
)

adjustment_sharpe = (
    adjustment_excess_returns.mean()
    / adjustment_excess_returns.std()
    * np.sqrt(trading_days)
)

adjustment_comparison["Sharpe Ratio"] = (
    adjustment_comparison["Portfolio"]
    .map(adjustment_sharpe)
)

display(
    adjustment_comparison.style
    .format({
        "Annualized Return": "{:.2%}",
        "Annualized Volatility": "{:.2%}",
        "Maximum Drawdown": "{:.2%}",
        "Daily VaR (95%)": "{:.2%}",
        "Daily ES (95%)": "{:.2%}",
        "Sharpe Ratio": "{:.2f}"
    })
    .hide(axis="index")
    .set_caption(
        "Original vs. Adjusted Resilient Portfolio"
    )
)

Portfolio,Annualized Return,Annualized Volatility,Maximum Drawdown,Daily VaR (95%),Daily ES (95%),Sharpe Ratio
Original Resilient,6.48%,9.00%,-25.77%,0.81%,1.32%,0.59
Adjusted Resilient,6.80%,9.22%,-26.27%,0.83%,1.36%,0.61


In [14]:
# Interest-rate sensitivity comparison

portfolio_value = 100000
bnd_duration = 5.9
basis_point = 0.0001

rate_sensitivity_comparison = pd.DataFrame({
    "Portfolio": [
        "Original Resilient",
        "Adjusted Resilient"
    ],
    "BND Weight": [
        resilient_original["BND"],
        resilient_adjusted["BND"]
    ]
})

rate_sensitivity_comparison["Portfolio DV01"] = (
    portfolio_value
    * rate_sensitivity_comparison["BND Weight"]
    * bnd_duration
    * basis_point
)

display(
    rate_sensitivity_comparison.style
    .format({
        "BND Weight": "{:.0%}",
        "Portfolio DV01": "${:.2f}"
    })
    .hide(axis="index")
    .set_caption(
        "Interest-Rate Sensitivity Comparison"
    )
)

Portfolio,BND Weight,Portfolio DV01
Original Resilient,45%,$26.55
Adjusted Resilient,35%,$20.65


#### Interpretation
Reducing the BND allocation lowers the Resilient portfolio's measured bond-duration exposure. Under the first-order duration approximation, the adjusted portfolio would experience a smaller valuation impact through its BND position for a given change in yields.

This calculation isolates BND's modeled interest-rate sensitivity rather than the complete rate exposure of every asset in the portfolio. BIL has some interest-rate sensitivity, and the other assets may also respond to changing rates through different economic channels.

The historical comparison shows that the adjustment modestly increases annualized return but also produces slightly higher volatility, maximum drawdown, VaR, and Expected Shortfall. Reducing one concentration therefore does not necessarily reduce every measure of portfolio risk.

The next step is to evaluate whether the reduction in BND exposure provides a meaningful benefit under the rate-driven stress environments that motivated the adjustment.

#### Targeted Rate-Stress Evaluation

Because the proposed adjustment was designed specifically to reduce interest-rate sensitivity, its effectiveness is evaluated under a historical environment characterized by substantial increases in interest rates.

The 2022 inflation and monetary-tightening episode provides a relevant test because Treasury yields increased sharply while equity markets also declined. The objective is to determine whether the adjusted Resilient portfolio provides meaningful protection during a rate-driven stress environment relative to the original allocation.

The comparison uses the January 3–December 30, 2022 close-to-close period from Notebook 5 rather than the shorter January–October event window used in Notebook 4.

In [15]:
# 2022 inflation and interest-rate stress period
stress_2022_start = pd.Timestamp("2022-01-03")
stress_2022_end = pd.Timestamp("2022-12-30")

# Match the same close-to-close period used in Notebook 5
resilient_2022_returns = resilient_comparison_returns.loc[
    (resilient_comparison_returns.index > stress_2022_start)
    & (resilient_comparison_returns.index <= stress_2022_end)
]

# Cumulative portfolio returns
resilient_2022_stress = (
    (1 + resilient_2022_returns).prod() - 1
)

resilient_2022_stress = pd.DataFrame({
    "Portfolio": resilient_2022_stress.index,
    "2022 Stress Return": resilient_2022_stress.values
})

display(
    resilient_2022_stress.style
    .format({
        "2022 Stress Return": "{:.2%}"
    })
    .hide(axis="index")
    .set_caption(
        "Original vs. Adjusted Resilient Portfolio — 2022 Rate Stress"
    )
)

Portfolio,2022 Stress Return
Original Resilient,-12.64%
Adjusted Resilient,-11.33%


#### Interpretation
The adjusted Resilient portfolio experienced a smaller loss than the original allocation during the selected 2022 stress period.

The improvement reflects the combined effect of reducing BND and increasing gold and short-term Treasury exposure. It is consistent with the objective of reducing bond-duration concentration, although the performance difference cannot be attributed to interest-rate sensitivity alone because multiple asset weights changed.

The result provides evidence of an improvement in this particular historical environment rather than a general reduction in portfolio risk.

#### Targeted Stagflation Stress Evaluation

The adjusted allocation is also evaluated under the Severe Stagflation scenario developed in Notebook 5. This scenario produced the largest hypothetical loss for the original Resilient portfolio and therefore provides a second targeted test of whether the proposed allocation change improves resilience to the risks that motivated the adjustment.

In [16]:
# Severe Stagflation shocks from Notebook 5
stagflation_shocks = pd.read_csv(
    n5_output_dir / "stagflation_shocks.csv",
    index_col=0
)["Shock"]

# Calculate portfolio responses
original_stagflation = sum(
    resilient_original[asset] * stagflation_shocks[asset]
    for asset in stagflation_shocks.index
)

adjusted_stagflation = sum(
    resilient_adjusted[asset] * stagflation_shocks[asset]
    for asset in stagflation_shocks.index
)

stagflation_comparison = pd.DataFrame({
    "Portfolio": [
        "Original Resilient",
        "Adjusted Resilient"
    ],
    "Severe Stagflation Return": [
        original_stagflation,
        adjusted_stagflation
    ]
})

display(
    stagflation_comparison.style
    .format({
        "Severe Stagflation Return": "{:.2%}"
    })
    .hide(axis="index")
    .set_caption(
        "Original vs. Adjusted Resilient Portfolio — Severe Stagflation"
    )
)

Portfolio,Severe Stagflation Return
Original Resilient,-15.90%
Adjusted Resilient,-13.60%


#### Allocation Decision
The adjusted Resilient portfolio addresses the specific risk concentration identified during the review. Reducing BND lowers the measured bond-duration exposure, while the revised allocation experiences smaller losses during the selected 2022 rate-stress period and the Severe Stagflation scenario.

These improvements come with trade-offs. Across the full historical sample, the adjusted portfolio produces slightly higher annualized return but also modestly higher volatility, maximum drawdown, VaR, and Expected Shortfall. The adjustment therefore does not improve every measure of risk.

For this project, the adjusted Resilient allocation is selected because it reduces a clearly identified bond-duration concentration while preserving the original 40% equity allocation and overall defensive structure. The original allocation remains a reasonable alternative when minimizing the observed full-sample downside measures is the primary consideration.

This decision is based on historical and hypothetical scenario evidence rather than an optimization process or out-of-sample validation. The results support the adjustment as a targeted portfolio-management decision, not as evidence that it will outperform the original allocation in future rising-rate environments.

## Part IV — Portfolio Monitoring and Rebalancing Framework
The portfolio-management framework establishes how the selected strategic allocations would be monitored and maintained over time.

It distinguishes between routine risk monitoring, allocation-based rebalancing, and strategic review. Changes in portfolio risk generate review signals, while trades are considered only after evaluating whether an intervention is consistent with the portfolio's objective.

### Monitoring Schedule and Review Process

In [17]:
monitoring_framework = pd.DataFrame({
    "Review Type": [
        "Routine Monitoring",
        "Formal Portfolio Review",
        "Strategic Allocation Review",
        "Event-Driven Review"
    ],

    "Frequency": [
        "Monthly",
        "Quarterly",
        "Annually",
        "As needed"
    ],

    "Primary Focus": [
        "Allocation drift, volatility, drawdown, and major market developments",
        "Risk-return performance, VaR/ES, factor sensitivities, and stress exposure",
        "Strategic weights, portfolio objectives, and long-term risk trade-offs",
        "Material market shocks or meaningful changes in portfolio risk"
    ],

    "Potential Action": [
        "Continue monitoring or flag issues for review",
        "Rebalance if allocation limits are breached",
        "Consider strategic allocation changes if justified",
        "Reassess exposures and determine whether intervention is warranted"
    ]
})

display(
    monitoring_framework.style
    .hide(axis="index")
    .set_caption(
        "Portfolio Monitoring and Review Framework"
    )
)

Review Type,Frequency,Primary Focus,Potential Action
Routine Monitoring,Monthly,"Allocation drift, volatility, drawdown, and major market developments",Continue monitoring or flag issues for review
Formal Portfolio Review,Quarterly,"Risk-return performance, VaR/ES, factor sensitivities, and stress exposure",Rebalance if allocation limits are breached
Strategic Allocation Review,Annually,"Strategic weights, portfolio objectives, and long-term risk trade-offs",Consider strategic allocation changes if justified
Event-Driven Review,As needed,Material market shocks or meaningful changes in portfolio risk,Reassess exposures and determine whether intervention is warranted


### Allocation Rebalancing Policy
The portfolios use a hybrid calendar and tolerance-band framework. Formal reviews occur quarterly, while material allocation drift or changes in risk conditions may trigger additional reviews.

For this project, the allowable deviation from an asset's strategic target is defined using a 5/25 rule: Each asset receives a tolerance equal to the smaller of five percentage points or 25% of its strategic target weight:

$$
\text{Tolerance Band}
=
\min(0.05,\;0.25\times w_i)
$$

where $w_i$ represents the strategic target weight of asset $i$.

A tolerance-band breach triggers a review rather than an unconditional trade. Before rebalancing, the portfolio manager would consider the source of the deviation, market conditions, liquidity, transaction costs, and whether the original strategic allocation remains appropriate. The resulting lower and upper bounds establish the range within which an asset allocation may fluctuate without requiring a formal rebalancing review. 
These bands are illustrative policy assumptions rather than universal industry-prescribed limits.

#### Rebalancing Tolerance Bands

The following table applies the tolerance policy to the three active allocations, using the adjusted Resilient portfolio selected in Part III.


In [18]:
# Active strategic allocations used for monitoring

active_allocations = {
    "Aggressive": weights_aggressive,
    "Balanced": weights_balanced,
    "Resilient": resilient_adjusted
}

# Calculate 5/25 tolerance bands
rebalancing_rows = []

for portfolio, weights in active_allocations.items():

    for asset, target_weight in weights.items():

        # Smaller of 5 percentage points or 25% of target weight
        tolerance = min(
            0.05,
            0.25 * target_weight
        )

        lower_band = max(
            0,
            target_weight - tolerance
        )

        upper_band = min(
            1,
            target_weight + tolerance
        )

        rebalancing_rows.append({
            "Portfolio": portfolio,
            "Asset": asset,
            "Target Weight": target_weight,
            "Tolerance": tolerance,
            "Lower Band": lower_band,
            "Upper Band": upper_band
        })

rebalancing_bands = pd.DataFrame(
    rebalancing_rows
)

display(
    rebalancing_bands.style
    .format({
        "Target Weight": "{:.2%}",
        "Tolerance": "±{:.2%}",
        "Lower Band": "{:.2%}",
        "Upper Band": "{:.2%}"
    })
    .hide(axis="index")
    .set_caption(
        "Strategic Allocation Rebalancing Bands"
    )
)

Portfolio,Asset,Target Weight,Tolerance,Lower Band,Upper Band
Aggressive,VTI,55.00%,±5.00%,50.00%,60.00%
Aggressive,VEU,20.00%,±5.00%,15.00%,25.00%
Aggressive,VWO,10.00%,±2.50%,7.50%,12.50%
Aggressive,BND,5.00%,±1.25%,3.75%,6.25%
Aggressive,GLD,5.00%,±1.25%,3.75%,6.25%
Aggressive,BIL,5.00%,±1.25%,3.75%,6.25%
Balanced,VTI,40.00%,±5.00%,35.00%,45.00%
Balanced,VEU,15.00%,±3.75%,11.25%,18.75%
Balanced,VWO,10.00%,±2.50%,7.50%,12.50%
Balanced,BND,25.00%,±5.00%,20.00%,30.00%


#### Interpretation

The tolerance-band framework allows larger strategic positions to experience moderate market-driven drift while applying tighter proportional controls to smaller allocations. This prevents relatively small portfolio positions from moving excessively away from their intended role before triggering review.

The bands are used as review triggers for potential rebalancing rather than automatic trading instructions. When an allocation moves outside its permitted range, the portfolio would be reviewed to determine whether the deviation reflects ordinary market movement, a broader change in risk conditions, or a structural issue requiring intervention.

This approach helps maintain the intended strategic characteristics of each portfolio while limiting unnecessary trading caused by small short-term fluctuations in asset values.

### Rebalancing Breach Monitoring
The following exercise demonstrates how the tolerance-band framework could be applied to the active portfolios.

Each portfolio is assumed to begin the current quarter at its strategic target allocation. No contributions, withdrawals, or trades occur during the monitoring period, allowing the asset weights to drift according to their realized returns.

The resulting weights are compared with the predetermined tolerance bands to identify allocations that would require review. The exercise is an illustrative implementation of the monitoring policy rather than a record of actual portfolio trading.

In [19]:
# Identify the final available date
latest_date = asset_returns.index.max()

# Determine the previous calendar quarter-end
current_quarter_start = latest_date.to_period("Q").start_time
rebalance_date = current_quarter_start - pd.Timedelta(days=1)

print(f"Assumed rebalance date: {rebalance_date.date()}")
print(f"End of monitoring period: {latest_date.date()}")

Assumed rebalance date: 2026-06-30
End of monitoring period: 2026-09-18


In [20]:
# Asset returns after the assumed quarterly rebalance
drift_period_returns = asset_returns.loc[
    (asset_returns.index > rebalance_date)
    & (asset_returns.index <= latest_date)
]

drift_rows = []

for portfolio, weights in active_allocations.items():

    target_weights = pd.Series(weights)

    # Cumulative growth of each asset during the monitoring period
    growth_factors = (
        1 + drift_period_returns[target_weights.index]
    ).prod()

    # Value of each position after market-driven drift
    ending_values = target_weights * growth_factors

    # Convert ending values back into portfolio weights
    ending_weights = ending_values / ending_values.sum()

    for asset in target_weights.index:

        drift_rows.append({
            "Portfolio": portfolio,
            "Asset": asset,
            "Target Weight": target_weights[asset],
            "End-of-Period Weight": ending_weights[asset]
        })

portfolio_drift = pd.DataFrame(drift_rows)

In [21]:
# Add tolerance-band limits
portfolio_drift = portfolio_drift.merge(
    rebalancing_bands[
        [
            "Portfolio",
            "Asset",
            "Lower Band",
            "Upper Band"
        ]
    ],
    on=["Portfolio", "Asset"],
    how="left"
)

# Calculate allocation drift in percentage points
portfolio_drift["Drift (pp)"] = (
    portfolio_drift["End-of-Period Weight"]
    - portfolio_drift["Target Weight"]
) * 100

# Flag tolerance-band breaches
portfolio_drift["Status"] = np.where(
    (
        portfolio_drift["End-of-Period Weight"]
        < portfolio_drift["Lower Band"]
    )
    |
    (
        portfolio_drift["End-of-Period Weight"]
        > portfolio_drift["Upper Band"]
    ),
    "Review Required",
    "Within Band"
)

display(
    portfolio_drift.style
    .format({
        "Target Weight": "{:.2%}",
        "End-of-Period Weight": "{:.2%}",
        "Lower Band": "{:.2%}",
        "Upper Band": "{:.2%}",
        "Drift (pp)": "{:+.2f}"
    })
    .hide(axis="index")
    .set_caption(
        "Quarterly Allocation Drift and Rebalancing Review"
    )
)

Portfolio,Asset,Target Weight,End-of-Period Weight,Lower Band,Upper Band,Drift (pp),Status
Aggressive,VTI,55.00%,55.07%,50.00%,60.00%,+0.07,Within Band
Aggressive,VEU,20.00%,19.81%,15.00%,25.00%,-0.19,Within Band
Aggressive,VWO,10.00%,9.94%,7.50%,12.50%,-0.06,Within Band
Aggressive,BND,5.00%,4.84%,3.75%,6.25%,-0.16,Within Band
Aggressive,GLD,5.00%,5.37%,3.75%,6.25%,+0.37,Within Band
Aggressive,BIL,5.00%,4.97%,3.75%,6.25%,-0.03,Within Band
Balanced,VTI,40.00%,40.30%,35.00%,45.00%,+0.30,Within Band
Balanced,VEU,15.00%,14.95%,11.25%,18.75%,-0.05,Within Band
Balanced,VWO,10.00%,10.00%,7.50%,12.50%,+0.00,Within Band
Balanced,BND,25.00%,24.33%,20.00%,30.00%,-0.67,Within Band


In [22]:
from IPython.display import display, Markdown

breaches = portfolio_drift[
    portfolio_drift["Status"] == "Review Required"
]

as_of_date = latest_date.strftime("%B %d, %Y")

if breaches.empty:
    monitoring_result = (
        f"As of **{as_of_date}**, all monitored allocations "
        f"remain within their established tolerance bands. "
        f"No allocation-based rebalancing review is triggered "
        f"by the simulated quarter-to-date drift."
    )
else:
    monitoring_result = (
        f"As of **{as_of_date}**, "
        f"**{len(breaches)} asset allocation(s)** have moved "
        f"outside their established tolerance bands and "
        f"would require review before any trading decision."
    )

display(Markdown(
    f"**Monitoring Result**\n\n{monitoring_result}"
))

**Monitoring Result**

As of **September 18, 2026**, all monitored allocations remain within their established tolerance bands. No allocation-based rebalancing review is triggered by the simulated quarter-to-date drift.

#### Interpretation
The tolerance-band framework allows portfolio weights to respond to ordinary market movements without generating automatic trading instructions.

A breach indicates that the portfolio's current allocation should be reviewed. The decision to rebalance would depend on the magnitude and source of the drift, the broader risk environment, and whether the strategic targets remain appropriate.

Risk concerns may also trigger a separate event-driven review even when all allocations remain within their permitted ranges.

## Part V — Final Portfolio Assessment and Recommendation
The final assessment brings together the project's historical performance, downside-risk measures, stress tests, factor sensitivities, and allocation decisions.

Each strategy is evaluated according to its intended role and remaining risk exposures rather than a single performance measure.

### Final Portfolio Comparison
The purpose of the comparison is not to rank the portfolios from best to worst, but to determine the role each strategy is best suited to serve and whether its remaining risks are consistent with that role.

The comparison below presents the three active strategies: the original Aggressive and Balanced portfolios and the adjusted Resilient portfolio selected in Part III.

In [23]:
# Historical metrics for Aggressive and Balanced
final_metrics = risk_summary[
    risk_summary["Portfolio"].isin(
        ["Aggressive", "Balanced"]
    )
].copy()

# Add the adjusted Resilient portfolio metrics
adjusted_final = adjustment_comparison[
    adjustment_comparison["Portfolio"]
    == "Adjusted Resilient"
].copy()

adjusted_final["Portfolio"] = "Resilient"

# Combine the three active portfolios
final_metrics = pd.concat(
    [
        final_metrics,
        adjusted_final
    ],
    ignore_index=True
)

# Add qualitative portfolio-management assessment
final_metrics["Strategic Role"] = [
    "Long-term growth",
    "Balanced growth and downside protection",
    "Capital preservation and defensive growth"
]

final_metrics["Primary Strength"] = [
    "Highest historical return",
    "Balanced risk-return characteristics",
    "Lowest historical downside-risk measures"
]

final_metrics["Primary Risk"] = [
    "Equity-driven volatility and drawdown",
    "Exposure to both equity and interest-rate risk",
    "Lower growth potential and remaining rate sensitivity"
]

final_metrics["Final Assessment"] = [
    "Maintain",
    "Maintain",
    "Maintain adjusted allocation"
]

display(
    final_metrics.style
    .format({
        "Annualized Return": "{:.2%}",
        "Annualized Volatility": "{:.2%}",
        "Maximum Drawdown": "{:.2%}",
        "Daily VaR (95%)": "{:.2%}",
        "Daily ES (95%)": "{:.2%}",
        "Sharpe Ratio": "{:.2f}"
    })
    .hide(axis="index")
    .set_caption(
        "Final Strategic Portfolio Comparison"
    )
)

Portfolio,Annualized Return,Annualized Volatility,Maximum Drawdown,Daily VaR (95%),Daily ES (95%),Sharpe Ratio,Strategic Role,Primary Strength,Primary Risk,Final Assessment
Aggressive,8.47%,17.43%,-50.09%,1.59%,2.66%,0.47,Long-term growth,Highest historical return,Equity-driven volatility and drawdown,Maintain
Balanced,7.43%,13.56%,-39.49%,1.23%,2.04%,0.49,Balanced growth and downside protection,Balanced risk-return characteristics,Exposure to both equity and interest-rate risk,Maintain
Resilient,6.80%,9.22%,-26.27%,0.83%,1.36%,0.61,Capital preservation and defensive growth,Lowest historical downside-risk measures,Lower growth potential and remaining rate sensitivity,Maintain adjusted allocation


#### Interpretation
The final comparison shows that each portfolio serves a different strategic objective.

Aggressive Growth produced the highest historical return but also experienced the greatest volatility, drawdown, and vulnerability to severe equity-led stress. Balanced Growth retained meaningful equity exposure while reducing downside risk relative to Aggressive.

The adjusted Resilient portfolio maintained the lowest historical downside-risk measures among the three active strategies while reducing the bond-duration concentration identified in the original allocation. The adjustment improved performance under the targeted 2022 and stagflation stress environments, although it modestly increased several full-sample risk measures relative to the original Resilient strategy.

The risk-adjusted results provide an additional perspective: the strategy with the highest historical return was not necessarily the one with the highest return relative to volatility.

These differences reinforce the importance of evaluating each allocation against its intended objective rather than relying on a single measure of performance or risk. The appropriate strategy depends on the investor's required balance between long-term growth, tolerance for drawdowns, and need for protection against adverse market conditions.

### Final Recommendation
The analysis supports maintaining the Aggressive and Balanced strategic allocations while adopting the adjusted Resilient allocation for the final portfolio-management framework.

Aggressive Growth remains aligned with an objective focused on long-term capital appreciation and acceptance of substantial equity-driven volatility and drawdown risk. Its higher equity allocation provides the greatest historical return potential but also creates the greatest sensitivity to severe equity-market stress.

Balanced Growth remains aligned with an objective that combines meaningful equity-market participation with a greater emphasis on downside protection. This makes it the most suitable allocation when neither maximum growth nor maximum defensiveness is the dominant objective.

The adjusted Resilient portfolio is selected for the defensive mandate because it reduces the measured BND duration concentration and improves outcomes under the targeted rate-driven stress environments. This decision accepts modestly higher historical volatility and downside-risk measures relative to the original Resilient allocation.

The choice among these strategies depends on the intended risk mandate. All three allocations would be managed through the monitoring, stress-testing, and tolerance-band framework developed in Part IV.

## Project Conclusion
This project developed an end-to-end portfolio risk-management framework for three strategic allocations with different growth and downside-protection objectives. The analysis progressed from data preparation and portfolio construction to historical performance measurement, scenario analysis, factor-sensitivity modeling, stress testing, reverse stress testing, and portfolio-management decisions.

The results demonstrate a consistent trade-off between historical return and downside protection. Aggressive Growth generated the highest long-term return but experienced the greatest volatility and losses during severe equity-market stress. Balanced Growth provided an intermediate risk-return profile, while Resilient Growth emphasized downside protection with lower equity sensitivity.

An important finding is that reducing one source of risk can increase exposure to another. The original Resilient portfolio reduced equity-related losses but had greater bond-duration sensitivity because of its larger BND allocation. A targeted adjustment reduced that concentration and improved results under selected rate-driven stress conditions, although it modestly increased several full-sample risk measures. This illustrates why portfolio resilience must be evaluated across different economic environments rather than through a single historical statistic.

The final framework connects those findings to portfolio-management decisions. Strategic allocations are evaluated against their intended objectives, changing risk exposures are monitored, and tolerance-band breaches trigger review rather than automatic trading. The project demonstrates how quantitative risk analysis can support allocation decisions while recognizing the trade-offs, assumptions, and limitations involved in managing portfolio risk.